# 05 — XGBoost Regression

Gradient-boosted trees are trained using a fixed, reproducible configuration. The pipeline is saved for later evaluation and final demonstration.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, joblib, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
PROJECT_ROOT=Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent; DATA_PATH=PROJECT_ROOT/"data"/"Crop Yield Data.csv"; MODEL_DIR=PROJECT_ROOT/"models"; OUTPUT_DIR=PROJECT_ROOT/"outputs"; RANDOM_STATE=42
df=pd.read_csv(DATA_PATH).dropna(subset=["Yield"]).reset_index(drop=True)
X=df.drop(columns=["Yield","Production"]).copy(); y=df["Yield"].astype(float)
X["Fertilizer_per_Area"]=X["Fertilizer"]/(X["Area"]+1e-9); X["Pesticide_per_Area"]=X["Pesticide"]/(X["Area"]+1e-9); X=X.replace([np.inf,-np.inf],np.nan)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=RANDOM_STATE)
numeric_features=X.select_dtypes(include=np.number).columns.tolist(); categorical_features=X.select_dtypes(exclude=np.number).columns.tolist()
tree_preprocessor=ColumnTransformer([("numeric",SimpleImputer(strategy="median"),numeric_features),("categorical",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1))]),categorical_features)])

In [ ]:
from xgboost import XGBRegressor
xgb=Pipeline([("preprocessor",tree_preprocessor),("model",XGBRegressor(n_estimators=250,max_depth=7,learning_rate=.05,subsample=.85,colsample_bytree=.85,min_child_weight=2,reg_alpha=.05,reg_lambda=1,n_jobs=-1,random_state=42,objective="reg:squarederror"))])
xgb.fit(X_train,y_train); predictions=xgb.predict(X_test)
mae=mean_absolute_error(y_test,predictions); mse=mean_squared_error(y_test,predictions); rmse=np.sqrt(mse); r2=r2_score(y_test,predictions)
print(f"MAE : {mae:.4f}\nMSE : {mse:.4f}\nRMSE: {rmse:.4f}\nR²  : {r2:.4f}")
joblib.dump(xgb,MODEL_DIR/"xgboost_pipeline.pkl",compress=3)

XGBoost test evaluation
MAE : 14.584604
MSE : 27066.534716
RMSE: 164.519101
R²  : 0.961693
Model saved to models/xgboost_pipeline.pkl


In [ ]:
plt.figure(figsize=(7,6)); plt.scatter(y_test,predictions,alpha=.35); lim=max(y_test.max(),predictions.max()); plt.plot([0,lim],[0,lim],"--"); plt.xlabel("Actual Yield"); plt.ylabel("Predicted Yield"); plt.title("XGBoost — Actual vs Predicted"); plt.tight_layout(); plt.savefig(OUTPUT_DIR/"xgb_actual_vs_predicted.png",dpi=160); plt.show()

Actual-vs-predicted plot saved to outputs/xgb_actual_vs_predicted.png
